"""
Structural causality (nominal→nominal) pipeline for your snippet CSV.

Input CSV columns (as you showed):
person, article_index, snippet_index, snippet, gender

Outputs:

Relations emitted include: CAUSED, CAUSAL_PREP, RESPONSIBLE_FOR, CREDITED_FOR, and ACCUSED_OF (for 'accuse X of ...' allegations).
- edges per snippet: cause_text, effect_text, relation_type, confidence, evidence
- optional aggregation per article_index

Design:
1) Parse snippet with spaCy (sentences + dependencies + entities + noun_chunks).
2) Extract candidate nominal spans (PERSON/ORG + noun chunks + nominalizations).
3) High-recall candidate generation using dependency patterns:
   - causative verbs: cause/lead/prompt/trigger/force/spark/result
   - causal preps: because of/due to/as a result of
   - responsibility verbs: blame/accuse/hold responsible/credit
4) Filter candidates with a classifier (optional):
   - HuggingFace Transformers sequence classifier fine-tuned for causal relations
   - If you have a SemEval-trained model path, plug it in.
   - If not, run in "pattern-only" mode.

Notes:
- This is snippet-level; you can later aggregate by article_index.
- Patterns are intentionally conservative structurally, but still high recall.
"""

In [1]:
from __future__ import annotations

import re
import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, List, Optional, Tuple, Dict

import pandas as pd
import spacy

# Optional: Transformers classifier
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_OK = True
except Exception:
    TRANSFORMERS_OK = False

In [2]:
# -----------------------------
# Config / Lexicons
# -----------------------------

# Split causative triggers into:
# - STRONG: usually causal without needing a specific complement
# - WEAK: polysemous; require a specific construction and/or event-like effect
STRONG_CAUSATIVE_VERBS = {
    "cause", "trigger", "force", "prompt", "spark"
}

WEAK_CAUSATIVE_VERBS = {
    "lead", "result", "bring", "produce"
}

# Required complements for weak verbs when used causally
# (spaCy uses 'prep' edges; we check the preposition lemma)
WEAK_VERB_REQUIRED_PREP = {
    "lead": {"to", "into"},
    "result": {"in", "into"},
    # "bring" is safest in "bring about" (or occasionally "bring to" / "bring into")
    "bring": {"about"},
    # "produce" often takes a direct object; we handle via event-like effect filter
    "produce": set(),
}

# Responsibility / attribution verbs (Entman-aligned)
RESP_ATTRIB_VERBS = {
    "blame", "accuse", "credit", "praise", "fault", "hold"
}

# Some verbs (especially "hold") are extremely ambiguous; only keep if in an attribution construction
# (e.g., "hold X responsible for Y")
HOLD_ATTRIB_KEYWORDS = {"responsible", "accountable", "culpable"}

# Stoplist for effects that are almost always non-outcome in this genre or are too generic/artefactual.
BAD_EFFECT_LEMMAS = {
    "hair", "weather", "breakfast", "article", "office", "home", "area", "country",
    "government", "people", "day", "time", "year", "morning", "week", "month", "date",
    "thing", "way", "place"
}

# A lightweight "event/outcome" head lemma list. Extend per your domain.
EVENTLIKE_EFFECT_HEADS = {
    "crisis", "backlash", "failure", "resignation", "defeat", "surge", "shock", "impasse",
    "division", "victory", "condemnation", "collapse", "increase", "decline", "harm",
    "damage", "uproar", "anger", "wrath", "tremor", "warning", "assault", "attack",
    "debate", "referendum", "vote", "u-turn", "windfall", "legislation", "programme",
    "policy", "deal", "transition", "delay", "holdup", "delay", "problem", "issue"
}

# Causal-preposition markers (detected as token windows here)
CAUSAL_PREP_MARKERS = ("because of", "due to", "as a result of")

NOMINALIZATION_SUFFIXES = ("tion", "sion", "ment", "ance", "ence", "ability", "ality", "ness", "ship")


In [3]:
# -----------------------------
# Data structures
# -----------------------------

@dataclass
class SpanRef:
    """A lightweight reference to a nominal span."""
    text: str
    start: int  # token start
    end: int    # token end (exclusive)
    head_i: int # head token index
    label: str  # e.g., PERSON/ORG/NP/EVENT/OUTCOME/ANCHOR

@dataclass
class CausalEdge:
    person: str
    article_index: int
    snippet_index: int
    relation: str           # CAUSED / ENABLED / RESPONSIBLE_FOR / CREDITED_FOR / CAUSAL_PREP / CONDITIONAL
    cause: str
    effect: str
    confidence: float
    evidence: str           # small string explaining which pattern fired
    sent_id: int            # sentence index within snippet


In [4]:
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s.strip())
    return s

def token_span_text(doc, start: int, end: int) -> str:
    return doc[start:end].text

def is_nominalization(token) -> bool:
    t = token.lemma_.lower()
    return any(t.endswith(suf) for suf in NOMINALIZATION_SUFFIXES)

def safe_softmax_conf(logits) -> float:
    # binary classifier assumed; return P(positive)
    # Works even if logits are a list/tuple
    if isinstance(logits, (list, tuple)):
        logits = logits[0]
    if hasattr(logits, "detach"):
        logits = logits.detach().cpu().float().tolist()
    if len(logits) == 1:
        return 1 / (1 + math.exp(-logits[0]))
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]
    probs = [e / sum(exps) for e in exps]
    # convention: positive class is index 1
    return float(probs[1]) if len(probs) > 1 else float(probs[0])



In [5]:
class PairClassifier:
    """
    Optional pair filter.
    Expect a binary classifier: [NOT_CAUSAL, CAUSAL] or similar.
    If you have a multi-class model, adapt predict_proba().
    """
    def __init__(self, model_name_or_path: str, device: Optional[str] = None):
        if not TRANSFORMERS_OK:
            raise RuntimeError("Transformers not available. pip install transformers torch")
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
        self.model.to(self.device)
        self.model.eval()

    def score(self, text: str) -> float:
        """
        Returns P(causal) in [0,1].
        """
        batch = self.tok(
            text,
            truncation=True,
            max_length=256,
            padding=False,
            return_tensors="pt"
        )
        batch = {k: v.to(self.device) for k, v in batch.items()}
        with torch.no_grad():
            out = self.model(**batch)
        return safe_softmax_conf(out.logits[0])


def format_pair_for_model(snippet_text: str, cause: str, effect: str) -> str:
    """
    Simple marker format for a pair classifier.
    If your SemEval fine-tuned model expects <e1> <e2>, keep that.
    """
    return f"[TEXT] {snippet_text} [CAUSE] {cause} [/CAUSE] [EFFECT] {effect} [/EFFECT]"



In [6]:
def extract_nominals(doc, anchor_person: str) -> List[SpanRef]:
    """
    Extracts:
    - Anchor node (best-effort string match)
    - Named entities (PERSON/ORG/GPE)
    - Noun chunks (NP)
    - Some event/outcome-like nominals (nominalizations)
    """
    spans: List[SpanRef] = []

    # 1) Named entities
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE"}:
            spans.append(SpanRef(ent.text, ent.start, ent.end, ent.root.i, ent.label_))

    # 2) Noun chunks
    for nc in doc.noun_chunks:
        # Drop very short pronouns/determiners
        if nc.root.pos_ == "PRON":
            continue
        text = normalize_text(nc.text)
        if len(text) < 3:
            continue
        spans.append(SpanRef(text, nc.start, nc.end, nc.root.i, "NP"))

    # 3) Nominalizations (single tokens promoted to spans if not already covered)
    covered = {(s.start, s.end) for s in spans}
    for tok in doc:
        if tok.pos_ in {"NOUN", "PROPN"} and is_nominalization(tok):
            key = (tok.i, tok.i + 1)
            if key not in covered:
                spans.append(SpanRef(tok.text, tok.i, tok.i + 1, tok.i, "NOMINALIZATION"))

    # 4) Anchor node: try to match last name / full name in text
    anchor_person_norm = anchor_person.lower().strip()
    parts = [p for p in anchor_person_norm.split() if p]
    variants = set()
    variants.add(anchor_person_norm)
    if parts:
        variants.add(parts[-1])  # last name
    # Find earliest match span
    anchor_span = None
    doc_lower = doc.text.lower()
    for v in sorted(variants, key=len, reverse=True):
        m = re.search(r"\b" + re.escape(v) + r"\b", doc_lower)
        if m:
            # approximate token span via char_to_token
            start_tok = doc.char_span(m.start(), m.end(), alignment_mode="expand")
            if start_tok:
                anchor_span = start_tok
                break
    if anchor_span:
        spans.append(SpanRef(anchor_span.text, anchor_span.start, anchor_span.end, anchor_span.root.i, "ANCHOR"))
    else:
        # If no match, still create a pseudo-node (won't have token alignment)
        spans.append(SpanRef(anchor_person, -1, -1, -1, "ANCHOR"))

    # Deduplicate by (start,end,text,label) preferring more specific labels
    priority = {"ANCHOR": 0, "PERSON": 1, "ORG": 2, "GPE": 3, "NOMINALIZATION": 4, "NP": 5}
    dedup: Dict[Tuple[int, int, str], SpanRef] = {}
    for s in spans:
        key = (s.start, s.end, s.text)
        if key not in dedup:
            dedup[key] = s
        else:
            if priority.get(s.label, 99) < priority.get(dedup[key].label, 99):
                dedup[key] = s
    return list(dedup.values())


def find_span_by_token(spans: List[SpanRef], tok_i: int) -> Optional[SpanRef]:
    """
    Returns the smallest span that contains tok_i.
    """
    candidates = [s for s in spans if s.start <= tok_i < s.end and s.start >= 0]
    if not candidates:
        return None
    # smallest by length
    return sorted(candidates, key=lambda s: (s.end - s.start, s.label))[0]



In [7]:
# -----------------------------
# Candidate generation via dependency patterns
# -----------------------------

def sentence_index_map(doc) -> Dict[int, int]:
    """
    token.i -> sentence_id
    """
    m = {}
    for si, sent in enumerate(doc.sents):
        for tok in sent:
            m[tok.i] = si
    return m


In [8]:
def extract_edges_by_patterns(doc, spans: List[SpanRef], person: str, article_index: int, snippet_index: int) -> List[CausalEdge]:
    """
    High-recall, structural causal candidate extraction.
    Produces directed edges (cause→effect) with a pattern-derived relation and heuristic confidence.

    Key refinements vs the initial version:
    - Strong vs weak causative verbs (weak verbs require specific complements and/or event-like effects)
    - Event/outcome filtering on the EFFECT head (to reduce absurd edges like "Boris Johnson → all the hair")
    - Extra disambiguation for 'hold' so we only keep 'hold X responsible for Y' style attributions
    """
    edges: List[CausalEdge] = []
    tok2sent = sentence_index_map(doc)

    def add_edge(rel, cause_span, effect_span, conf, evidence, head_tok_i):
        edges.append(
            CausalEdge(
                person=person,
                article_index=int(article_index),
                snippet_index=int(snippet_index),
                relation=rel,
                cause=normalize_text(cause_span.text),
                effect=normalize_text(effect_span.text),
                confidence=float(conf),
                evidence=evidence,
                sent_id=int(tok2sent.get(head_tok_i, 0)),
            )
        )

    def span_head_lemma(span: SpanRef) -> str:
        if span.head_i is None or span.head_i < 0 or span.head_i >= len(doc):
            return ""
        return doc[span.head_i].lemma_.lower()

    def is_eventlike_effect(span: SpanRef) -> bool:
        """Heuristic: keep effects that look like outcomes/events, drop generic artefacts."""
        if span.start < 0:
            return False
        head = doc[span.head_i] if 0 <= span.head_i < len(doc) else None
        if head is None:
            return False
        lemma = head.lemma_.lower()

        if lemma in BAD_EFFECT_LEMMAS:
            return False
        # Drop pure pronoun effects
        if head.pos_ == "PRON":
            return False
        # Keep if in curated list
        if lemma in EVENTLIKE_EFFECT_HEADS:
            return True
        # Keep if the head itself is a nominalization (common for outcomes)
        if is_nominalization(head):
            return True
        # Keep if span label suggests it's an event-like nominalization
        if span.label in {"NOMINALIZATION"}:
            return True
        # As a fallback, allow some NPs that contain outcome-ish keywords
        text_l = span.text.lower()
        if any(k in text_l for k in ("crisis", "backlash", "failure", "resignation", "defeat", "surge", "shock", "impasse", "division", "victory", "condemnation")):
            return True
        return False

    def is_plausible_cause(span: SpanRef) -> bool:
        """Heuristic cause sanity filter: drop pronouns and non-nominal heads (often parser artefacts)."""
        if span.start < 0:
            return True  # allow anchor-only pseudo node
        head = doc[span.head_i] if 0 <= span.head_i < len(doc) else None
        if head is None:
            return False

        # Reject WH-possessive fragments like "whose lies" (often unhelpful as standalone causes)
        first_tok = doc[span.start] if 0 <= span.start < len(doc) else None
        if first_tok is not None and first_tok.lower_ in {"whose"}:
            return False

        # Drop pronouns as heads
        if head.pos_ == "PRON":
            return False
        # For structural causality between nominals, require the head to be nominal-ish.
        # This removes junk spans whose head is a VERB/ADV/etc. (e.g., weird lemmas from tokenization).
        if head.pos_ not in {"NOUN", "PROPN"} and not is_nominalization(head):
            return False
        return True


    # -------------------------
    # Pattern family 1: causative verbs
    # -------------------------
    for tok in doc:
        if tok.pos_ != "VERB":
            continue
        lemma = tok.lemma_.lower()

        # Active/passive arguments
        nsubj = next((c for c in tok.children if c.dep_ in {"nsubj", "nsubj:pass"}), None)
        dobj = next((c for c in tok.children if c.dep_ in {"dobj", "obj", "attr"}), None)

        # Prepositional complements, e.g. "lead to X", "result in X", "bring about X"
        preps = [c for c in tok.children if c.dep_ == "prep"]
        prep = preps[0] if preps else None

        pobj = None
        if prep:
            pobj = next((c for c in prep.children if c.dep_ in {"pobj", "obj"}), None)

        # Passive "X was caused by Y"
        by_prep = next((c for c in tok.children if c.dep_ == "prep" and c.lemma_.lower() == "by"), None)
        by_obj = None
        if by_prep:
            by_obj = next((c for c in by_prep.children if c.dep_ in {"pobj", "obj"}), None)

        # Decide whether this verb instance is "causal enough" to consider
        if lemma in STRONG_CAUSATIVE_VERBS:
            pass  # allow (still filtered by event-like effect)
        elif lemma in WEAK_CAUSATIVE_VERBS:
            # For weak verbs, enforce complement constraints
            required_preps = WEAK_VERB_REQUIRED_PREP.get(lemma, set())
            if required_preps:
                # must have the right prep (e.g., lead→to, result→in, bring→about)
                if not (prep and prep.lemma_.lower() in required_preps and pobj is not None):
                    continue
            else:
                # e.g., produce: allow direct object but will require event-like effect
                if dobj is None and pobj is None:
                    continue
        else:
            continue  # not in our causality trigger list

        # Map to spans and emit edges
        # Active: nsubj -> cause, (dobj or pobj) -> effect
        if nsubj and (dobj or pobj):
            cause = find_span_by_token(spans, nsubj.i)
            eff_tok_i = (dobj.i if dobj else pobj.i)
            effect = find_span_by_token(spans, eff_tok_i)
            if cause and effect and cause.text != effect.text and is_plausible_cause(cause) and is_eventlike_effect(effect):
                # Confidence: strong verbs higher than weak verbs
                base = 0.75 if lemma in STRONG_CAUSATIVE_VERBS else 0.60
                # Boost if we matched a canonical weak construction (lead to / result in / bring about)
                if lemma in WEAK_CAUSATIVE_VERBS and prep is not None and pobj is not None:
                    base += 0.05
                add_edge("CAUSED", cause, effect, base, f"causative_verb:{lemma}", tok.i)

        # Passive: nsubj:pass is effect, by_obj is cause
        if by_obj and nsubj:
            cause = find_span_by_token(spans, by_obj.i)
            effect = find_span_by_token(spans, nsubj.i)
            if cause and effect and cause.text != effect.text and is_plausible_cause(cause) and is_eventlike_effect(effect):
                base = 0.75 if lemma in STRONG_CAUSATIVE_VERBS else 0.60
                add_edge("CAUSED", cause, effect, base, f"passive_causative_verb:{lemma}", tok.i)

    # -------------------------
    # Pattern family 2: causal preposition markers
    # -------------------------
    text_lower = doc.text.lower()

    def pick_effect_span_before(marker_char_start: int, marker_char_end: int) -> Optional[SpanRef]:
        """Pick an effect span structurally (prefer event/outcome tied to the main predicate), falling back to nearest nominal."""
        # Convert char position to token position
        marker_span = doc.char_span(marker_char_start, marker_char_end, alignment_mode="expand")
        if marker_span is None:
            return None
        marker_tok_i = marker_span.start

        # 1) Look left for a main VERB and take its object/attr as the effect nominal if possible.
        for i in range(marker_tok_i - 1, -1, -1):
            t = doc[i]
            if t.pos_ == "VERB" and t.dep_ not in {"aux", "auxpass"}:
                # Prefer direct object / attr / pobj of a prep complement
                obj = next((c for c in t.children if c.dep_ in {"dobj", "obj", "attr"}), None)
                if obj is not None:
                    sp = find_span_by_token(spans, obj.i)
                    if sp and is_eventlike_effect(sp):
                        return sp
                # If no object, sometimes the effect is a nominal subject complement; try attribute children
                # Otherwise fall through to nominal fallback
                break

        # 2) Fallback: nearest nominal span ending before the marker (old heuristic)
        effect_candidates = [
            s for s in spans
            if s.label in {"NP", "NOMINALIZATION", "ORG", "PERSON", "GPE"}
            and s.start >= 0
            and doc[s.end - 1].idx < marker_char_start
        ]
        if not effect_candidates:
            return None
        return sorted(effect_candidates, key=lambda s: doc[s.end - 1].idx, reverse=True)[0]

    def pick_cause_span_after(marker_char_start: int, marker_char_end: int) -> Optional[SpanRef]:
        """Pick a cause span as the first nominal after the marker."""
        cause_candidates = [
            s for s in spans
            if s.label in {"NP", "NOMINALIZATION", "ORG", "PERSON", "GPE"}
            and s.start >= 0
            and doc[s.start].idx > marker_char_end
        ]
        if not cause_candidates:
            return None
        return sorted(cause_candidates, key=lambda s: doc[s.start].idx)[0]

    for marker in CAUSAL_PREP_MARKERS:
        if marker not in text_lower:
            continue
        for m in re.finditer(r"" + re.escape(marker) + r"", text_lower):
            effect = pick_effect_span_before(m.start(), m.end())
            cause = pick_cause_span_after(m.start(), m.end())
            if effect is None or cause is None:
                continue

            # Filter: effect shouldn't be a PERSON, and must look event-like
            if effect.label == "PERSON":
                continue
            if not is_plausible_cause(cause):
                continue
            if not is_eventlike_effect(effect):
                continue

            add_edge("CAUSAL_PREP", cause, effect, 0.60, f"marker:{marker.replace(' ', '_')}", effect.head_i)


    # -------------------------
    # Pattern family 3: responsibility attribution verbs (blame/accuse/credit/praise/hold responsible)
    # -------------------------
    for tok in doc:
        if tok.pos_ != "VERB":
            continue
        lemma = tok.lemma_.lower()
        if lemma not in RESP_ATTRIB_VERBS:
            continue

        # Disambiguate "hold": keep only "hold X responsible/accountable/culpable ..."
        if lemma == "hold":
            # Look for an xcomp/attr/adjective with lemma responsible/accountable/culpable
            if not any((c.lemma_.lower() in HOLD_ATTRIB_KEYWORDS) for c in tok.children):
                # also allow "held responsible" where "responsible" is in the subtree
                subtree_lemmas = {t.lemma_.lower() for t in tok.subtree}
                if not (HOLD_ATTRIB_KEYWORDS & subtree_lemmas):
                    continue

        # "X blamed Y for Z" / "X accused Y of Z" / "X credited Y for Z"
        dobj = next((c for c in tok.children if c.dep_ in {"dobj", "obj"}), None)
        prep_for = next((c for c in tok.children if c.dep_ == "prep" and c.lemma_.lower() in {"for", "of", "with"}), None)
        pobj = None
        if prep_for:
            pobj = next((c for c in prep_for.children if c.dep_ in {"pobj", "obj"}), None)

        if dobj and pobj:
            blamed = find_span_by_token(spans, dobj.i)
            outcome = find_span_by_token(spans, pobj.i)

            # Special case: "accuse X of <clause>" where pobj is a VERB/AUX (e.g., "of being ...")
            clause_outcome = None
            if outcome is None and lemma == "accuse":
                # Prefer an xcomp/ccomp child of pobj if present; otherwise use pobj subtree
                clause_head = next((c for c in pobj.children if c.dep_ in {"xcomp", "ccomp"}), None) or pobj
                left = clause_head.left_edge.i
                right = clause_head.right_edge.i + 1
                clause_text = normalize_text(doc[left:right].text)
                # Keep it reasonably short to avoid swallowing whole paragraphs
                if 0 < len(clause_text) <= 140:
                    clause_outcome = SpanRef(
                        text=clause_text,
                        start=left,
                        end=right,
                        head_i=clause_head.i,
                        label="CLAUSE"
                    )

            if blamed and (outcome or clause_outcome):
                out = outcome or clause_outcome

                # For explicit responsibility/credit relations, keep the stronger event-like constraint.
                # For ACCUSED_OF, allow clause outcomes (allegations) and be more permissive.
                rel = "RESPONSIBLE_FOR"
                if lemma in {"credit", "praise"}:
                    rel = "CREDITED_FOR"
                elif lemma == "accuse":
                    rel = "ACCUSED_OF"

                if rel in {"RESPONSIBLE_FOR", "CREDITED_FOR"}:
                    if not is_eventlike_effect(out):
                        continue
                else:
                    # ACCUSED_OF: drop trivial/generic outcomes
                    out_head = doc[out.head_i] if out.head_i >= 0 else None
                    if out_head and out_head.lemma_.lower() in BAD_EFFECT_LEMMAS:
                        continue

                if normalize_text(blamed.text).lower() == normalize_text(out.text).lower():
                    continue

                add_edge(rel, blamed, out, 0.75, f"attrib_verb:{lemma}", tok.i)

    # Deduplicate edges (same cause/effect/relation)
    uniq = {}
    for e in edges:
        key = (e.person, e.article_index, e.snippet_index, e.relation, e.cause.lower(), e.effect.lower())
        if key not in uniq or uniq[key].confidence < e.confidence:
            uniq[key] = e
    return list(uniq.values())


In [9]:

# -----------------------------
# Pipeline runner
# -----------------------------

def run_pipeline(
    csv_path: str | Path,
    out_edges_csv: str | Path,
    spacy_model: str = "en_core_web_sm",
    classifier_model: Optional[str] = None,   # e.g. path to your SemEval fine-tuned model
    classifier_threshold: float = 0.55,
    pattern_only: bool = False
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    required = {"person", "article_index", "snippet_index", "snippet"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    nlp = spacy.load(spacy_model)
    # Ensure sentence boundaries
    if not nlp.pipe_names or "sentencizer" not in nlp.pipe_names:
        # sentencizer helps if the parser isn't enabled
        try:
            nlp.add_pipe("sentencizer")
        except Exception:
            pass

    clf = None
    if (not pattern_only) and classifier_model:
        clf = PairClassifier(classifier_model)

    all_edges: List[CausalEdge] = []

    # Batch parse for speed
    texts = df["snippet"].astype(str).tolist()
    docs = list(nlp.pipe(texts, batch_size=64))

    for row, doc in zip(df.itertuples(index=False), docs):
        person = str(row.person)
        aidx = int(row.article_index)
        sidx = int(row.snippet_index)
        snippet_text = str(row.snippet)

        spans = extract_nominals(doc, anchor_person=person)

        # Candidate edges by structural patterns
        edges = extract_edges_by_patterns(
            doc=doc,
            spans=spans,
            person=person,
            article_index=aidx,
            snippet_index=sidx
        )

        # Optional classifier filtering/scoring
        if clf is not None:
            kept = []
            for e in edges:
                model_input = format_pair_for_model(snippet_text, e.cause, e.effect)
                p = clf.score(model_input)
                # Combine pattern confidence and model confidence (simple product)
                combined = float(e.confidence * p)
                if combined >= classifier_threshold:
                    e.confidence = combined
                    e.evidence = f"{e.evidence}|clf"
                    kept.append(e)
            edges = kept

        all_edges.extend(edges)

    edges_df = pd.DataFrame([asdict(e) for e in all_edges])
    edges_df.to_csv(out_edges_csv, index=False)
    return edges_df


def aggregate_article_level(edges_df: pd.DataFrame) -> pd.DataFrame:
    """
    Example article-level aggregation: structural “causality strength” + most common targets.
    """
    if edges_df.empty:
        return pd.DataFrame(columns=["article_index", "person", "edge_count", "mean_conf", "max_conf"])

    agg = (edges_df
           .groupby(["article_index", "person"], as_index=False)
           .agg(edge_count=("confidence", "size"),
                mean_conf=("confidence", "mean"),
                max_conf=("confidence", "max")))
    return agg


# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    # 1) Pattern-only run (no ML dependency)
    edges = run_pipeline(
        csv_path="Brexit_Referendum_snippets.csv",
        out_edges_csv="causal_edges_snippet_level.csv",
        spacy_model="en_core_web_sm",
        classifier_model=None,
        pattern_only=True
    )
    print(edges.head(20))

    # 2) If you have a fine-tuned SemEval-style classifier:
    # edges = run_pipeline(
    #     csv_path="coreference_snippets.csv",
    #     out_edges_csv="causal_edges_snippet_level.csv",
    #     spacy_model="en_core_web_sm",
    #     classifier_model="path/to/your/semeval-causality-model",
    #     classifier_threshold=0.25,   # product conf; tune on dev set
    #     pattern_only=False
    # )

    art = aggregate_article_level(edges)
    art.to_csv("causality_article_level.csv", index=False)
    print(art.head(20))

           person  article_index  snippet_index         relation  \
0    michael gove             23              1           CAUSED   
1    nigel farage             24              6           CAUSED   
2   jeremy corbyn             49              5  RESPONSIBLE_FOR   
3   jeremy corbyn             49              6  RESPONSIBLE_FOR   
4     theresa may             82              5           CAUSED   
5     theresa may             82              6           CAUSED   
6     theresa may             82              7           CAUSED   
7     theresa may            116              1           CAUSED   
8      lisa nandy            135              1           CAUSED   
9      lisa nandy            135              2           CAUSED   
10    theresa may            235             13           CAUSED   
11  boris johnson            235             13           CAUSED   
12  boris johnson            235             14           CAUSED   
13  boris johnson            235             15 